## Two_Tower_BPR_AI_senmatics

Two-tower models are a popular architecture in recommender systems and information retrieval. They are designed to efficiently retrieve relevant items for a given user or query by learning separate, independent representations (embeddings) for users/queries and items.

### How it Works:

1.  **User Tower (Query Tower):** This tower takes user features (e.g., user ID, demographic information, past interactions, preferences) as input and outputs a dense vector embedding that represents the user's preferences or intent.

2.  **Item Tower (Candidate Tower):** This tower takes item features (e.g., item ID, descriptive attributes, content-based features like text embeddings) as input and outputs a dense vector embedding that represents the item.

### Key Characteristics:

*   **Efficiency:** Once the user and item embeddings are generated, the similarity between a user and all possible items can be computed very quickly (often via dot product or cosine similarity), making them suitable for large-scale retrieval.
*   **Scalability:** The two-tower structure allows for pre-computation of item embeddings, which can be stored in a vector database for fast nearest-neighbor search.
*   **Flexibility:** Different types of features (categorical, numerical, text, image) can be incorporated into each tower, often using neural networks to learn rich representations.

### Common Applications:

*   **Recommendation Systems:** Suggesting products, movies, music, or articles to users.
*   **Search Engines:** Ranking search results based on the query and document relevance.
*   **Ad Targeting:** Matching users with relevant advertisements.

In this notebook, we're building a multi-modal two-tower model where the user tower processes user IDs and aggregated historical interaction data. A key innovation in the item tower is its integration of various modalities like structured features, synopsis BERT embeddings, **AI-generated semantic tags**, and review BERT embeddings to create a comprehensive item representation. This unique combination allows for a richer understanding of item characteristics.

In [ ]:
import os
import copy
import random
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# ============================================================
# 1. Config
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

BASE_PATH = Path("/content/drive/MyDrive")
TRAIN_PATH = BASE_PATH / "train.csv"
VAL_PATH = BASE_PATH / "val.csv"
TEST_PATH = BASE_PATH / "test.csv"

# New master matrix with semantics
MASTER_ANIME_PATH = BASE_PATH / "MASTER_ANIME_TOWER_FEATURES_WITH_SEMANTICS.npy"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ----------------------------
# SETTINGS
# ----------------------------
POS_THRESHOLD = 8.0

BATCH_SIZE = 1024
EPOCHS = 5
LR = 3e-4
WEIGHT_DECAY = 1e-6
PATIENCE = 2

EMB_DIM = 64
HIDDEN_DIM = 192
DROPOUT = 0.10

MAX_TRAIN_POS_PER_USER = None

# Full-catalog evaluation settings
EVAL_K = 10
EVAL_MAX_USERS = 300   # set None if you want full validation/test, but slower

# ============================================================
# 2. Feature block boundaries (fixed from your assembler)
# ============================================================
STRUCT_START, STRUCT_END = 0, 65
SYN_START, SYN_END = 65, 449
TAG_START, TAG_END = 449, 510
REV_START, REV_END = 510, 894

STRUCT_DIM = STRUCT_END - STRUCT_START      # 65
SYN_DIM = SYN_END - SYN_START               # 384
TAG_DIM = TAG_END - TAG_START               # 61
REV_DIM = REV_END - REV_START               # 384
ITEM_FEAT_DIM_TO_USE = REV_END              # 894

print("=== FEATURE BLOCKS ===")
print(f"Structured   : [{STRUCT_START}:{STRUCT_END}] dim={STRUCT_DIM}")
print(f"Synopsis BERT: [{SYN_START}:{SYN_END}] dim={SYN_DIM}")
print(f"Semantic Tags: [{TAG_START}:{TAG_END}] dim={TAG_DIM}")
print(f"Review BERT  : [{REV_START}:{REV_END}] dim={REV_DIM}")
print(f"Total item dim: {ITEM_FEAT_DIM_TO_USE}")

# ============================================================
# 3. Load train / val / test
# ============================================================
train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VAL_PATH)
test_df = pd.read_csv(TEST_PATH)

if "score" in train_df.columns and "rating" not in train_df.columns:
    train_df = train_df.rename(columns={"score": "rating"})
if "score" in val_df.columns and "rating" not in val_df.columns:
    val_df = val_df.rename(columns={"score": "rating"})
if "score" in test_df.columns and "rating" not in test_df.columns:
    test_df = test_df.rename(columns={"score": "rating"})

required_cols = ["user_idx", "anime_idx", "rating"]
for c in required_cols:
    if c not in train_df.columns:
        raise ValueError(f"Missing column in train.csv: {c}")
    if c not in val_df.columns:
        raise ValueError(f"Missing column in val.csv: {c}")
    if c not in test_df.columns:
        raise ValueError(f"Missing column in test.csv: {c}")

train_df = train_df[required_cols].copy()
val_df = val_df[required_cols].copy()
test_df = test_df[required_cols].copy()

for df in [train_df, val_df, test_df]:
    df["user_idx"] = df["user_idx"].astype(int)
    df["anime_idx"] = df["anime_idx"].astype(int)
    df["rating"] = df["rating"].astype(float)

NUM_USERS = int(max(train_df.user_idx.max(), val_df.user_idx.max(), test_df.user_idx.max())) + 1
MAX_ANIME_IDX = int(max(train_df.anime_idx.max(), val_df.anime_idx.max(), test_df.anime_idx.max()))

print("Train:", train_df.shape, "Val:", val_df.shape, "Test:", test_df.shape)
print("NUM_USERS:", NUM_USERS)
print("MAX_ANIME_IDX:", MAX_ANIME_IDX)

# ============================================================
# 4. Load MASTER_ANIME features
# ============================================================
master_anime = np.load(MASTER_ANIME_PATH).astype(np.float32)
print("Original master_anime shape:", master_anime.shape)

# ============================================================
# 4. Load MASTER_ANIME features
# ============================================================
master_anime = np.load(MASTER_ANIME_PATH).astype(np.float32)
print("Original master_anime shape:", master_anime.shape)

expected_rows = MAX_ANIME_IDX + 1  # because anime_idx goes from 0 to MAX_ANIME_IDX

if master_anime.shape[0] < expected_rows:
    raise ValueError(
        f"MASTER_ANIME has only {master_anime.shape[0]} rows, "
        f"but split data requires at least {expected_rows} rows."
    )

# If master has more rows than needed, keep only the rows used by current split
if master_anime.shape[0] > expected_rows:
    print(f"MASTER_ANIME has extra rows. Truncating from {master_anime.shape[0]} to {expected_rows}.")
    master_anime = master_anime[:expected_rows]

master_anime = master_anime[:, :ITEM_FEAT_DIM_TO_USE]
NUM_ITEMS = master_anime.shape[0]

if master_anime.shape[1] != ITEM_FEAT_DIM_TO_USE:
    raise ValueError(
        f"Expected item feature dim {ITEM_FEAT_DIM_TO_USE}, got {master_anime.shape[1]}"
    )

master_anime = np.nan_to_num(master_anime, nan=0.0, posinf=0.0, neginf=0.0)

# Split feature blocks
item_struct = master_anime[:, STRUCT_START:STRUCT_END]
item_syn = master_anime[:, SYN_START:SYN_END]
item_tag = master_anime[:, TAG_START:TAG_END]
item_rev = master_anime[:, REV_START:REV_END]

# Normalize each block separately
def block_zscore(x):
    x = x.copy()
    mu = x.mean(axis=0, keepdims=True)
    sigma = x.std(axis=0, keepdims=True) + 1e-6
    return (x - mu) / sigma

item_struct = block_zscore(item_struct)
item_syn = block_zscore(item_syn)
item_tag = block_zscore(item_tag)
item_rev = block_zscore(item_rev)

print("Trimmed master_anime shape:", master_anime.shape)
print("item_struct shape:", item_struct.shape)
print("item_syn shape:", item_syn.shape)
print("item_tag shape:", item_tag.shape)
print("item_rev shape:", item_rev.shape)

master_anime = master_anime[:, :ITEM_FEAT_DIM_TO_USE]
NUM_ITEMS = master_anime.shape[0]

if master_anime.shape[1] != ITEM_FEAT_DIM_TO_USE:
    raise ValueError(
        f"Expected item feature dim {ITEM_FEAT_DIM_TO_USE}, got {master_anime.shape[1]}"
    )

master_anime = np.nan_to_num(master_anime, nan=0.0, posinf=0.0, neginf=0.0)

# Split feature blocks
item_struct = master_anime[:, STRUCT_START:STRUCT_END]
item_syn = master_anime[:, SYN_START:SYN_END]
item_tag = master_anime[:, TAG_START:TAG_END]
item_rev = master_anime[:, REV_START:REV_END]

# Normalize each block separately
def block_zscore(x, skip_first_row=True):
    x = x.copy()
    if x.shape[0] == 0:
        return x
    if skip_first_row and x.shape[0] > 1:
        mu = x[1:].mean(axis=0, keepdims=True)
        sigma = x[1:].std(axis=0, keepdims=True) + 1e-6
        x[1:] = (x[1:] - mu) / sigma
    else:
        mu = x.mean(axis=0, keepdims=True)
        sigma = x.std(axis=0, keepdims=True) + 1e-6
        x = (x - mu) / sigma
    return x

item_struct = block_zscore(item_struct, skip_first_row=True)
item_syn = block_zscore(item_syn, skip_first_row=True)
item_tag = block_zscore(item_tag, skip_first_row=True)
item_rev = block_zscore(item_rev, skip_first_row=True)

print("item_struct shape:", item_struct.shape)
print("item_syn shape:", item_syn.shape)
print("item_tag shape:", item_tag.shape)
print("item_rev shape:", item_rev.shape)

# ============================================================
# 5. Build user features from TRAIN ONLY
#    Keep this relatively simple to avoid overpowering text
# ============================================================
train_pos_df = train_df[train_df["rating"] >= POS_THRESHOLD].copy()
if len(train_pos_df) == 0:
    raise ValueError(f"No positive samples found in train.csv with POS_THRESHOLD={POS_THRESHOLD}")

if MAX_TRAIN_POS_PER_USER is not None:
    train_pos_df = (
        train_pos_df.groupby("user_idx", group_keys=False)
        .apply(lambda g: g.sample(n=min(len(g), MAX_TRAIN_POS_PER_USER), random_state=SEED))
        .reset_index(drop=True)
    )

user_stats_df = train_df.groupby("user_idx").agg(
    user_rating_count=("rating", "count"),
    user_rating_mean=("rating", "mean"),
    user_rating_std=("rating", "std")
).fillna(0.0)

# Use only structured item block to construct a stable user preference vector
item_pref_sum = np.zeros((NUM_USERS, STRUCT_DIM), dtype=np.float32)
item_pref_weight = np.zeros(NUM_USERS, dtype=np.float32)

for row in train_pos_df.itertuples(index=False):
    u = int(row.user_idx)
    i = int(row.anime_idx)
    r = float(row.rating)

    if i <= 0 or i >= NUM_ITEMS:
        continue

    w = max((r - POS_THRESHOLD + 1.0), 1.0)
    item_pref_sum[u] += w * item_struct[i]
    item_pref_weight[u] += w

user_pref_vec = np.zeros((NUM_USERS, STRUCT_DIM), dtype=np.float32)
mask = item_pref_weight > 0
user_pref_vec[mask] = item_pref_sum[mask] / item_pref_weight[mask][:, None]

user_basic = np.zeros((NUM_USERS, user_stats_df.shape[1]), dtype=np.float32)
user_basic[user_stats_df.index.values] = user_stats_df.values.astype(np.float32)

user_dense = np.concatenate([user_basic, user_pref_vec], axis=1)
user_dense = np.nan_to_num(user_dense, nan=0.0, posinf=0.0, neginf=0.0)

nz = (np.abs(user_dense).sum(axis=1) > 0)
if nz.any():
    mu = user_dense[nz].mean(axis=0, keepdims=True)
    sigma = user_dense[nz].std(axis=0, keepdims=True) + 1e-6
    user_dense[nz] = (user_dense[nz] - mu) / sigma

print("user_dense shape:", user_dense.shape)
print("train_pos_df shape after cap:", train_pos_df.shape)

# ============================================================
# 6. Prepare positives / seen maps
# ============================================================
user_seen_train = train_pos_df.groupby("user_idx")["anime_idx"].apply(set).to_dict()

val_pos_df = val_df[val_df["rating"] >= POS_THRESHOLD].copy()
val_pos_by_user = val_pos_df.groupby("user_idx")["anime_idx"].apply(set).to_dict()

test_pos_df = test_df[test_df["rating"] >= POS_THRESHOLD].copy()
test_pos_by_user = test_pos_df.groupby("user_idx")["anime_idx"].apply(set).to_dict()

print("Train positives:", train_pos_df.shape)
print("Val positives:", val_pos_df.shape)
print("Test positives:", test_pos_df.shape)

# ============================================================
# 7. BPR dataset
# ============================================================
class TwoTowerBPRDataset(Dataset):
    def __init__(self, pos_df, user_seen_map, num_items):
        self.users = pos_df["user_idx"].to_numpy(np.int64)
        self.pos_items = pos_df["anime_idx"].to_numpy(np.int64)
        self.user_seen = user_seen_map
        self.num_items = num_items

    def __len__(self):
        return len(self.users)

    def __getitem__(self, idx):
        u = int(self.users[idx])
        p = int(self.pos_items[idx])

        seen = self.user_seen.get(u, set())
        n = np.random.randint(1, self.num_items)
        while n in seen:
            n = np.random.randint(1, self.num_items)

        return (
            torch.tensor(u, dtype=torch.long),
            torch.tensor(p, dtype=torch.long),
            torch.tensor(n, dtype=torch.long)
        )

train_loader = DataLoader(
    TwoTowerBPRDataset(train_pos_df, user_seen_train, NUM_ITEMS),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=(DEVICE.type == "cuda")
)

# ============================================================
# 8. Evaluation helpers (FULL CATALOG)
# ============================================================
def _dcg_at_k(binary_hits):
    if len(binary_hits) == 0:
        return 0.0
    denom = np.log2(np.arange(2, len(binary_hits) + 2))
    return float((binary_hits / denom).sum())

@torch.no_grad()
def precompute_all_item_scores_matrix(model, item_batch_size=2048):
    model.eval()

    item_ids_np = np.arange(1, NUM_ITEMS, dtype=np.int64)
    item_vecs = []

    for start in range(0, len(item_ids_np), item_batch_size):
        batch_ids_np = item_ids_np[start:start + item_batch_size]
        batch_ids = torch.tensor(batch_ids_np, dtype=torch.long, device=DEVICE)
        batch_vecs = model.item_vector(batch_ids)
        item_vecs.append(batch_vecs)

    item_vecs = torch.cat(item_vecs, dim=0)
    return item_ids_np, item_vecs

@torch.no_grad()
def evaluate_full_catalog_at_k(model, pos_by_user, seen_items_by_user, k=10, max_users=300):
    if len(pos_by_user) == 0:
        return 0.0, 0.0, 0.0, 0.0

    model.eval()

    users = list(pos_by_user.keys())
    if max_users is not None and len(users) > max_users:
        users = list(np.random.choice(users, size=max_users, replace=False))

    item_ids_np, all_item_vecs = precompute_all_item_scores_matrix(model)

    precisions, recalls, hitrates, ndcgs = [], [], [], []
    user_batch_size = 256

    for start in range(0, len(users), user_batch_size):
        batch_users = users[start:start + user_batch_size]
        u_tensor = torch.tensor(batch_users, dtype=torch.long, device=DEVICE)

        user_vecs = model.user_vector(u_tensor)
        scores = torch.matmul(user_vecs, all_item_vecs.T).cpu().numpy()

        for row_idx, u in enumerate(batch_users):
            u = int(u)
            pos_items = np.array(list(pos_by_user.get(u, set())), dtype=np.int64)
            if len(pos_items) == 0:
                continue

            seen = seen_items_by_user.get(u, set())
            row_scores = scores[row_idx].copy()

            if len(seen) > 0:
                seen_mask = np.isin(item_ids_np, list(seen))
                row_scores[seen_mask] = -1e12

            top_idx = np.argpartition(row_scores, -k)[-k:]
            top_idx = top_idx[np.argsort(row_scores[top_idx])[::-1]]
            top_items = item_ids_np[top_idx]

            hits = np.isin(top_items, pos_items).astype(np.float32)
            num_hits = float(hits.sum())

            precisions.append(num_hits / k)
            recalls.append(num_hits / max(len(pos_items), 1))
            hitrates.append(1.0 if num_hits > 0 else 0.0)

            dcg = _dcg_at_k(hits)
            ideal_hits = np.ones(min(k, len(pos_items)), dtype=np.float32)
            idcg = _dcg_at_k(ideal_hits)
            ndcgs.append((dcg / idcg) if idcg > 0 else 0.0)

    if len(precisions) == 0:
        return 0.0, 0.0, 0.0, 0.0

    return (
        float(np.mean(precisions)),
        float(np.mean(recalls)),
        float(np.mean(hitrates)),
        float(np.mean(ndcgs))
    )

# ============================================================
# 9. Multi-modal Two-tower model
# ============================================================
class TwoTowerMultiModal(nn.Module):
    def __init__(self, num_users, num_items, user_dense_np, item_struct_np, item_syn_np, item_tag_np, item_rev_np):
        super().__init__()

        # user/item ID embeddings
        self.user_id_emb = nn.Embedding(num_users, EMB_DIM)
        self.item_id_emb = nn.Embedding(num_items, EMB_DIM)

        # feature tables
        self.register_buffer("user_dense_table", torch.tensor(user_dense_np, dtype=torch.float32))
        self.register_buffer("item_struct_table", torch.tensor(item_struct_np, dtype=torch.float32))
        self.register_buffer("item_syn_table", torch.tensor(item_syn_np, dtype=torch.float32))
        self.register_buffer("item_tag_table", torch.tensor(item_tag_np, dtype=torch.float32))
        self.register_buffer("item_rev_table", torch.tensor(item_rev_np, dtype=torch.float32))

        # user tower
        user_in_dim = EMB_DIM + self.user_dense_table.shape[1]
        self.user_mlp = nn.Sequential(
            nn.Linear(user_in_dim, HIDDEN_DIM),
            nn.ReLU(),
            nn.Dropout(DROPOUT),
            nn.Linear(HIDDEN_DIM, EMB_DIM)
        )

        # item branches
        self.item_struct_mlp = nn.Sequential(
            nn.Linear(STRUCT_DIM, 128),
            nn.ReLU(),
            nn.Dropout(DROPOUT),
            nn.Linear(128, 64)
        )

        self.item_syn_mlp = nn.Sequential(
            nn.Linear(SYN_DIM, 128),
            nn.ReLU(),
            nn.Dropout(DROPOUT),
            nn.Linear(128, 64)
        )

        self.item_tag_mlp = nn.Sequential(
            nn.Linear(TAG_DIM, 64),
            nn.ReLU(),
            nn.Dropout(DROPOUT),
            nn.Linear(64, 32)
        )

        self.item_rev_mlp = nn.Sequential(
            nn.Linear(REV_DIM, 128),
            nn.ReLU(),
            nn.Dropout(DROPOUT),
            nn.Linear(128, 64)
        )

        # lightweight modality gates
        self.gate_struct = nn.Linear(64, 1)
        self.gate_syn = nn.Linear(64, 1)
        self.gate_tag = nn.Linear(32, 1)
        self.gate_rev = nn.Linear(64, 1)

        # final fusion
        item_final_in = EMB_DIM + 64 + 64 + 32 + 64
        self.item_final_mlp = nn.Sequential(
            nn.Linear(item_final_in, HIDDEN_DIM),
            nn.ReLU(),
            nn.Dropout(DROPOUT),
            nn.Linear(HIDDEN_DIM, EMB_DIM)
        )

        # biases
        self.user_bias = nn.Embedding(num_users, 1)
        self.item_bias = nn.Embedding(num_items, 1)
        self.global_bias = nn.Parameter(torch.zeros(1))

        # init
        nn.init.normal_(self.user_id_emb.weight, std=0.02)
        nn.init.normal_(self.item_id_emb.weight, std=0.02)
        nn.init.zeros_(self.user_bias.weight)
        nn.init.zeros_(self.item_bias.weight)

    def user_vector(self, u_idx):
        x = torch.cat([self.user_id_emb(u_idx), self.user_dense_table[u_idx]], dim=1)
        return self.user_mlp(x)

    def item_vector(self, i_idx):
        id_vec = self.item_id_emb(i_idx)

        struct_vec = self.item_struct_mlp(self.item_struct_table[i_idx])
        syn_vec = self.item_syn_mlp(self.item_syn_table[i_idx])
        tag_vec = self.item_tag_mlp(self.item_tag_table[i_idx])
        rev_vec = self.item_rev_mlp(self.item_rev_table[i_idx])

        # modality gating
        struct_vec = torch.sigmoid(self.gate_struct(struct_vec)) * struct_vec
        syn_vec = torch.sigmoid(self.gate_syn(syn_vec)) * syn_vec
        tag_vec = torch.sigmoid(self.gate_tag(tag_vec)) * tag_vec
        rev_vec = torch.sigmoid(self.gate_rev(rev_vec)) * rev_vec

        x = torch.cat([id_vec, struct_vec, syn_vec, tag_vec, rev_vec], dim=1)
        return self.item_final_mlp(x)

    def score(self, u_idx, i_idx):
        u_vec = self.user_vector(u_idx)
        i_vec = self.item_vector(i_idx)
        dot = (u_vec * i_vec).sum(dim=1)
        return dot + self.user_bias(u_idx).squeeze(-1) + self.item_bias(i_idx).squeeze(-1) + self.global_bias

# ============================================================
# 10. Train
# ============================================================
model = TwoTowerMultiModal(
    num_users=NUM_USERS,
    num_items=NUM_ITEMS,
    user_dense_np=user_dense,
    item_struct_np=item_struct,
    item_syn_np=item_syn,
    item_tag_np=item_tag,
    item_rev_np=item_rev
).to(DEVICE)

optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

best_metric = -np.inf
best_state = None
best_epoch = -1
patience_counter = 0

print(f"Start training | users={NUM_USERS} | items={NUM_ITEMS} | POS_THRESHOLD={POS_THRESHOLD}")

for epoch in range(1, EPOCHS + 1):
    model.train()
    losses = []

    for u, p, n in train_loader:
        u = u.to(DEVICE, non_blocking=True)
        p = p.to(DEVICE, non_blocking=True)
        n = n.to(DEVICE, non_blocking=True)

        optimizer.zero_grad()

        pos_score = model.score(u, p)
        neg_score = model.score(u, n)

        loss = -F.logsigmoid(pos_score - neg_score).mean()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()

        losses.append(loss.item())

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    val_p, val_r, val_hr, val_ndcg = evaluate_full_catalog_at_k(
        model=model,
        pos_by_user=val_pos_by_user,
        seen_items_by_user=user_seen_train,
        k=EVAL_K,
        max_users=EVAL_MAX_USERS
    )

    print(
        f"Epoch {epoch:02d} | "
        f"bpr_loss={np.mean(losses):.4f} | "
        f"Val P@{EVAL_K}={val_p:.4f} | "
        f"Val R@{EVAL_K}={val_r:.4f} | "
        f"Val HR@{EVAL_K}={val_hr:.4f} | "
        f"Val NDCG@{EVAL_K}={val_ndcg:.4f}"
    )

    if val_ndcg > best_metric:
        best_metric = val_ndcg
        best_epoch = epoch
        best_state = copy.deepcopy(model.state_dict())
        patience_counter = 0
        print(f"  New best model at epoch {epoch}")
    else:
        patience_counter += 1
        print(f"  No improvement. Patience: {patience_counter}/{PATIENCE}")

    if patience_counter >= PATIENCE:
        print("Early stopping triggered.")
        break

if best_state is not None:
    model.load_state_dict(best_state)
    print(f"\nLoaded best model from epoch {best_epoch} with Val NDCG@{EVAL_K}={best_metric:.4f}")

# ============================================================
# 11. Final test evaluation (FULL CATALOG)
# ============================================================
test_p, test_r, test_hr, test_ndcg = evaluate_full_catalog_at_k(
    model=model,
    pos_by_user=test_pos_by_user,
    seen_items_by_user=user_seen_train,
    k=EVAL_K,
    max_users=EVAL_MAX_USERS
)

print("\nFinal Test Metrics (FULL CATALOG):")
print(f"P@{EVAL_K}   = {test_p:.4f}")
print(f"R@{EVAL_K}   = {test_r:.4f}")
print(f"HR@{EVAL_K}  = {test_hr:.4f}")
print(f"NDCG@{EVAL_K}= {test_ndcg:.4f}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Using device: cuda
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
=== FEATURE BLOCKS ===
Structured   : [0:65] dim=65
Synopsis BERT: [65:449] dim=384
Semantic Tags: [449:510] dim=61
Review BERT  : [510:894] dim=384
Total item dim: 894
Train: (25173563, 3) Val: (3152157, 3) Test: (3152157, 3)
NUM_USERS: 292565
MAX_ANIME_IDX: 13009
Original master_anime shape: (17562, 894)
Original master_anime shape: (17562, 894)
MASTER_ANIME has extra rows. Truncating from 17562 to 13010.
Trimmed master_anime shape: (13010, 894)
item_struct shape: (13010, 65)
item_syn shape: (13010, 384)
item_tag shape: (13010, 61)
item_rev shape: (13010, 384)
item_struct shape: (13010, 65)
item_syn shape: (13010, 384)
item_tag shape: (13010, 61)
item_rev shape: (13010, 384)
user_dense shape: (292565, 68)
train_pos_df shape after cap: (25043282, 3)
Train positives: (25043282, 3)
Val positi